# Manual multilabel training with cross-validation

Reserve the grouped test patients, select model settings and thresholds from grouped multilabel cross-validation, retrain on the full development pool, then evaluate the test set once.

In [ ]:
from copy import deepcopy
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader

In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / "configs/framework.yaml").exists():
    ROOT = ROOT.parent
if not (ROOT / "configs/framework.yaml").exists():
    raise FileNotFoundError("Run this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))

from src.data import (
    fit_standardizer,
    grouped_multilabel_folds,
    make_dataset,
    split_indices,
)
from src.hashing import calculate_run_id
from src.model import build_model
from src.training import (
    build_loss,
    build_optimizer,
    evaluate_epoch,
    predict_loader,
    resolve_final_epochs,
    save_artifacts,
    select_thresholds,
    train_epoch,
)

In [ ]:
framework_path = ROOT / "configs/framework.yaml"
framework_snapshot = framework_path.read_bytes()
config = yaml.safe_load(framework_snapshot)
project_path = ROOT / config["project"]["config_path"]
project_snapshot = project_path.read_bytes()
project_config = yaml.safe_load(project_snapshot)
config["project"].update(project_config)
project_basename = project_path.stem
run_id = calculate_run_id(config)
cv_config = config["cross_validation"]
if not cv_config.get("enabled"):
    raise ValueError("cross_validation.enabled must be true")
if cv_config["selection_metric"] != "micro_f1":
    raise ValueError("Only micro_f1 selection is currently supported")
print(f"project: {project_basename}, run_id: {run_id}")

In [ ]:
frame = pd.read_parquet(ROOT / config["data"]["path"])
feature_columns = project_config["features"]
label_columns = project_config["labels"]
missing_columns = set(feature_columns + label_columns) - set(frame.columns)
if missing_columns:
    raise ValueError(f"Columns not found in dataset: {sorted(missing_columns)}")
frame[feature_columns + label_columns].head()

## Reserve the grouped test set

The test rows are assigned once and are not loaded into a model dataset until final evaluation.

In [ ]:
split = config["split"]
train_idx, val_idx, test_idx = split_indices(
    len(frame),
    split["train"],
    split["val"],
    split["test"],
    split["seed"],
    frame=frame,
    label_columns=label_columns,
    strategy=split["strategy"],
    group_column=split["group_column"],
)
development_idx = np.sort(np.concatenate([train_idx, val_idx]))
development_frame = frame.iloc[development_idx].reset_index(drop=True)
development_patients = set(development_frame[split["group_column"]])
test_patients = set(frame.iloc[test_idx][split["group_column"]])
if not development_patients.isdisjoint(test_patients):
    raise RuntimeError("Patient leakage detected between development and test")

cv_folds = grouped_multilabel_folds(
    development_frame,
    label_columns,
    split["group_column"],
    cv_config["folds"],
    split["seed"],
)
pd.DataFrame(
    {
        "development": {
            "rows": len(development_idx),
            "patients": len(development_patients),
        },
        "test": {"rows": len(test_idx), "patients": len(test_patients)},
    }
).T

## Grouped multilabel cross-validation

Each candidate is trained fold-by-fold with early stopping. Out-of-fold probabilities select the candidate and decision thresholds.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
patience = config["training"]["early_stopping_patience"]
threshold_grid = [float(value) for value in cv_config["thresholds"]]
candidate_results = []
candidate_cache = {}

def with_candidate(base_config, candidate):
    candidate_config = deepcopy(base_config)
    for section in ("model", "training"):
        candidate_config[section].update(candidate.get(section, {}))
    return candidate_config

for candidate_index, candidate in enumerate(cv_config["candidates"]):
    print(f"\n=== candidate: {candidate['name']} ===")
    candidate_config = with_candidate(config, candidate)
    oof_probabilities = torch.zeros((len(development_frame), len(label_columns)))
    oof_targets = torch.zeros_like(oof_probabilities)
    oof_availability = torch.zeros_like(oof_probabilities)
    fold_results = []

    for fold_number, (fold_train_idx, fold_val_idx) in enumerate(cv_folds, start=1):
        fold_seed = split["seed"] + candidate_index * 100 + fold_number
        torch.manual_seed(fold_seed)
        means, scales = fit_standardizer(
            development_frame.iloc[fold_train_idx], feature_columns
        )
        fold_train_data = make_dataset(
            development_frame.iloc[fold_train_idx],
            feature_columns,
            label_columns,
            means,
            scales,
        )
        fold_val_data = make_dataset(
            development_frame.iloc[fold_val_idx],
            feature_columns,
            label_columns,
            means,
            scales,
        )
        generator = torch.Generator().manual_seed(fold_seed)
        fold_train_loader = DataLoader(
            fold_train_data,
            batch_size=candidate_config["training"]["batch_size"],
            shuffle=True,
            generator=generator,
        )
        fold_val_loader = DataLoader(
            fold_val_data,
            batch_size=candidate_config["training"]["batch_size"],
        )
        fold_model = build_model(candidate_config, project_config).to(device)
        fold_loss = build_loss(candidate_config["loss"]).to(device)
        fold_optimizer = build_optimizer(fold_model, candidate_config["training"])
        best_state = None
        best_val_loss = float("inf")
        best_epoch = 0
        epochs_without_improvement = 0

        for epoch in range(1, candidate_config["training"]["epochs"] + 1):
            train_metrics = train_epoch(
                fold_model, fold_train_loader, fold_loss, fold_optimizer, device
            )
            val_metrics = evaluate_epoch(fold_model, fold_val_loader, fold_loss, device)
            if best_state is None or val_metrics["loss"] < best_val_loss:
                best_state = deepcopy(fold_model.state_dict())
                best_val_loss = val_metrics["loss"]
                best_epoch = epoch
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
            print(
                f"candidate {candidate['name']} fold {fold_number} epoch {epoch:03d} | "
                f"train {train_metrics['loss']:.4f} | val {val_metrics['loss']:.4f} "
                f"f1 {val_metrics['micro_f1']:.3f} | "
                f"patience {epochs_without_improvement}/{patience}"
            )
            if epochs_without_improvement >= patience:
                break

        fold_model.load_state_dict(best_state)
        fold_logits, fold_targets, fold_masks = predict_loader(
            fold_model, fold_val_loader, device
        )
        oof_probabilities[fold_val_idx] = torch.sigmoid(fold_logits)
        oof_targets[fold_val_idx] = fold_targets
        oof_availability[fold_val_idx] = fold_masks
        fold_results.append(
            {
                "fold": fold_number,
                "best_epoch": best_epoch,
                "best_val_loss": float(best_val_loss),
            }
        )

    global_threshold, label_thresholds, threshold_scores = select_thresholds(
        oof_probabilities,
        oof_targets,
        oof_availability,
        threshold_grid,
        cv_config["min_positive_count_for_label_threshold"],
    )
    candidate_result = {
        "name": candidate["name"],
        "candidate": candidate,
        "micro_f1": float(threshold_scores[global_threshold]),
        "global_threshold": float(global_threshold),
        "threshold_scores": {
            str(threshold): float(score) for threshold, score in threshold_scores.items()
        },
        "folds": fold_results,
    }
    candidate_results.append(candidate_result)
    candidate_cache[candidate["name"]] = {
        "probabilities": oof_probabilities,
        "targets": oof_targets,
        "availability": oof_availability,
        "label_thresholds": label_thresholds,
    }

cv_results = pd.DataFrame(
    [
        {
            "candidate": result["name"],
            "oof_micro_f1": result["micro_f1"],
            "global_threshold": result["global_threshold"],
            "fold_best_epochs": [fold["best_epoch"] for fold in result["folds"]],
        }
        for result in candidate_results
    ]
).set_index("candidate")
cv_results

## Selected configuration and thresholds

In [ ]:
selected_result = max(candidate_results, key=lambda result: result["micro_f1"])
selected_candidate = selected_result["candidate"]
selected_config = with_candidate(config, selected_candidate)
selected_cache = candidate_cache[selected_result["name"]]
global_threshold = selected_result["global_threshold"]
label_thresholds = selected_cache["label_thresholds"]
fold_best_epochs = [fold["best_epoch"] for fold in selected_result["folds"]]
final_epochs = resolve_final_epochs(
    cv_config["final_epochs"], fold_best_epochs
)

threshold_table = pd.DataFrame(
    {
        "threshold": label_thresholds.numpy(),
        "development_positive_count": (
            (selected_cache["targets"] == 1) & selected_cache["availability"].bool()
        ).sum(dim=0).numpy(),
    },
    index=label_columns,
)
print(
    f"selected {selected_result['name']} at global threshold {global_threshold:.2f}\n"
    f"final epoch rule: {cv_config['final_epochs']}; fold best epochs: "
    f"{fold_best_epochs} -> retrain for {final_epochs} epochs"
)
display(cv_results, threshold_table.sort_values("development_positive_count").head(20))

## Retrain on all development patients, then evaluate test once

In [ ]:
means, scales = fit_standardizer(development_frame, feature_columns)
development_data = make_dataset(
    development_frame, feature_columns, label_columns, means, scales
)
test_data = make_dataset(
    frame.iloc[test_idx], feature_columns, label_columns, means, scales
)
final_generator = torch.Generator().manual_seed(split["seed"])
development_loader = DataLoader(
    development_data,
    batch_size=selected_config["training"]["batch_size"],
    shuffle=True,
    generator=final_generator,
)
test_loader = DataLoader(
    test_data, batch_size=selected_config["training"]["batch_size"]
)

torch.manual_seed(split["seed"])
model = build_model(selected_config, project_config).to(device)
criterion = build_loss(selected_config["loss"]).to(device)
optimizer = build_optimizer(model, selected_config["training"])
history = []
for epoch in range(1, final_epochs + 1):
    train_metrics = train_epoch(model, development_loader, criterion, optimizer, device)
    history.append(
        {"epoch": epoch, **{f"train_{key}": value for key, value in train_metrics.items()}}
    )
    print(
        f"final epoch {epoch:03d}/{final_epochs} | loss {train_metrics['loss']:.4f} "
        f"f1 {train_metrics['micro_f1']:.3f} acc {train_metrics['accuracy']:.3f}"
    )

test_metrics = evaluate_epoch(
    model, test_loader, criterion, device, threshold=label_thresholds
)
test_metrics

In [ ]:
metrics = {
    "project": project_basename,
    "run_id": run_id,
    "best_epoch": final_epochs,
    "epochs_trained": final_epochs,
    "stopped_early": False,
    "final_train": history[-1],
    "cross_validation": {
        "enabled": True,
        "folds": cv_config["folds"],
        "selection_metric": cv_config["selection_metric"],
        "selected_candidate": selected_candidate,
        "selected_global_threshold": float(global_threshold),
        "label_thresholds": {
            label: float(threshold)
            for label, threshold in zip(label_columns, label_thresholds)
        },
        "fold_best_epochs": fold_best_epochs,
        "candidate_results": candidate_results,
        "final_epochs": final_epochs,
    },
    "test": test_metrics,
}
output_dir = save_artifacts(
    run_id,
    project_basename,
    model,
    framework_snapshot,
    project_snapshot,
    metrics,
    history,
    ROOT / "artifacts",
)
print(f"Saved artifacts to {output_dir}")
metrics